In [8]:
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#from calcSplitRatio import calcSplitRatio
import calcSplitRatio as csr
csr.np = np

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

# ==========================================
# 1. ENVIRONMENT & DATA LOADING (POLARS)
# ==========================================
# Get current working directory (.\\notebooks or similar)
cwd = os.getcwd()

# Get parent folder to access (.\\Data)
parent = os.path.dirname(cwd)

# Get file path to load into environment
file_path_EPTI_data = os.path.join(parent, "Data", "EPTI_data.parquet")

print(f"Loading dataset from: {file_path_EPTI_data}")
df = pl.read_parquet(file_path_EPTI_data)
df.describe()

Loading dataset from: c:\Users\Marty\Documents\School\Merrimack College\8) DSE6311 Data Science Capstone\team_omega\Data\EPTI_data.parquet


statistic,ID,From,Date,Hour,Demand_kWh,Price_profile,Price_level,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72
str,str,str,str,f64,f64,str,str,f64,f64,f64,f64,f64
"""count""","""12180000""","""12180000""","""12180000""",1.218e7,1.218e7,"""1460760""","""1460760""",1.46076e6,1.218e7,1.218e7,1.218e7,1.218e7
"""null_count""","""0""","""0""","""0""",0.0,0.0,"""10719240""","""10719240""",1.071924e7,0.0,0.0,0.0,0.0
"""mean""",null,"""2021-01-27 23:29:59.999000+00:…","""2021-01-27 12:00:00""",12.5,2.564245,null,null,3.545642,-0.739941,-0.748828,-0.758594,-0.771649
"""std""",null,null,null,6.922187,1.99542,null,null,4.390327,5.124421,4.8099,4.67145,4.563512
"""min""","""Exp_1""","""2020-12-01 00:00:00+00:00""","""2020-12-01""",1.0,0.0,"""A""","""10""",0.0,-21.8,-18.1,-17.3,-17.0
"""25%""",null,"""2020-12-30 00:00:00+00:00""","""2020-12-30""",7.0,1.043,null,null,1.0,-4.3,-4.5,-4.5,-4.4
"""50%""",null,"""2021-01-28 00:00:00+00:00""","""2021-01-28""",13.0,2.151,null,null,1.1,0.0,0.3,0.4,0.4
"""75%""",null,"""2021-02-25 23:00:00+00:00""","""2021-02-25""",18.0,3.608,null,null,5.0,2.9,2.7,2.6,2.6
"""max""","""Exp_99""","""2021-03-26 23:00:00+00:00""","""2021-03-26""",24.0,22.62,"""P0""","""5_10""",30.15,12.8,10.6,9.7,9.1


In [9]:
csr.calcSplitRatio(df)

The ideal split ratio is 0.7:0.3 training:testing


np.float64(0.7)

In [19]:
# Ensure chronological order
df = df.sort("From")

# Calculate lag-1 autocorrelation using pl.corr()
autocorr = df.select(
    pl.corr(pl.col("Demand_kWh"), pl.col("Demand_kWh").shift(1))
).item()

print(f"Lag-1 Autocorrelation of Demand_kWh: {autocorr:.4f}")
if abs(autocorr) > 0.3:
    print("-> High temporal autocorrelation detected. Random shuffling is not recommended; use blocked or group splits.")

Lag-1 Autocorrelation of Demand_kWh: 0.2400


In [12]:
def assign_blocked_stratified_splits(
    df: pl.DataFrame, 
    date_col: str, 
    temp_col: str, 
    block_days: int = 7
) -> pl.DataFrame:
    # 1. Sort chronologically
    df = df.sort(date_col)
    
    # 2. Segment timeline into multi-day blocks (e.g., week-long runs)
    # Convert timestamps to epoch days, then integer division by block length
    df = df.with_columns(
        block_id=(pl.col(date_col).cast(pl.Date).dt.epoch() / (86400 * block_days)).cast(pl.Int64)
    )
    
    # 3. Aggregate temperature characteristics per block for stratification
    block_summary = (
        df.group_by("block_id")
        .agg(pl.col(temp_col).mean().alias("mean_temp"))
        .sort("mean_temp")
    )
    
    # Assign blocks cyclically or stratified across splits (60% Train, 20% Val, 20% Test)
    unique_blocks = block_summary.get_column("block_id").to_list()
    
    train_blocks, val_blocks, test_blocks = [], [], []
    for i, b_id in enumerate(unique_blocks):
        remainder = i % 5
        if remainder in [0, 1, 2]:  # 60% Train
            train_blocks.append(b_id)
        elif remainder == 3:        # 20% Validation
            val_blocks.append(b_id)
        else:                       # 20% Test
            test_blocks.append(b_id)
            
    # 4. Tag each row with its designated split while preserving intra-block continuity
    df = df.with_columns(
        split=pl.when(pl.col("block_id").is_in(train_blocks)).then(pl.lit("train"))
                .when(pl.col("block_id").is_in(val_blocks)).then(pl.lit("val"))
                .otherwise(pl.lit("test"))
    )
    
    return df

# Example usage:
partitioned_df = assign_blocked_stratified_splits(df, date_col="From", temp_col="Temperature")

In [15]:
partitioned_df.head()

ID,From,Date,Hour,Demand_kWh,Price_profile,Price_level,Experiment_price_NOK_kWh,Temperature,Temperature24,Temperature48,Temperature72,block_id,split
str,"datetime[ms, UTC]",date,i32,f64,str,str,f64,f64,f64,f64,f64,i64,str
"""Exp_43""",2020-12-01 00:00:00 UTC,2020-12-01,1,0.48,null,null,null,4.2,6.4,6.3,5.7,2656714285,"""test"""
"""Exp_286""",2020-12-01 00:00:00 UTC,2020-12-01,1,1.49,null,null,null,4.2,6.4,6.3,5.7,2656714285,"""test"""
"""Exp_539""",2020-12-01 00:00:00 UTC,2020-12-01,1,2.362,null,null,null,4.2,6.4,6.3,5.7,2656714285,"""test"""
"""Exp_552""",2020-12-01 00:00:00 UTC,2020-12-01,1,0.775,null,null,null,4.2,6.4,6.3,5.7,2656714285,"""test"""
"""Exp_36""",2020-12-01 00:00:00 UTC,2020-12-01,1,4.714,null,null,null,4.2,6.4,6.3,5.7,2656714285,"""test"""
